# Prática Guiada: Prevendo o Custo de um Plano de Saúde

**Pós-graduação PUC Minas | Machine Learning | Encontro 2**

Nesta prática vocês vão treinar sozinhos os modelos de regressão que vimos na aula teórica, usando uma base de dados real e mais simples do que a da aula anterior: custos de plano de saúde nos Estados Unidos, com idade, sexo, IMC, número de filhos e se a pessoa é fumante.

**Como funciona esta prática:**

- As primeiras etapas (carregar os dados, explorar, preparar) já vêm prontas, exatamente como fizemos juntos na aula anterior. Rodem essas células para seguir em frente.
- A partir da etapa de treinar os modelos, é a vez de vocês. Cada exercício traz um passo a passo em texto e uma célula de código com trechos marcados `# COMPLETE AQUI`, onde vocês escrevem o código.
- Sempre que precisarem, tem uma dica logo abaixo do exercício.

**Objetivo**: prever a coluna `charges` (custo do plano de saúde, em dólares) a partir das demais colunas.

**Antes de começar**: baixem o arquivo `custo_medico.csv` e deixem na mesma pasta deste notebook.

## 0. Preparando o Ambiente

Como sempre, começamos importando as bibliotecas que vamos usar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

## Etapa 1: Carregando e Conhecendo os Dados

Esta etapa já vem pronta, é a mesma sequência que fizemos juntos na aula anterior: carregar, olhar o formato, os tipos de cada coluna e checar valores faltando.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Substitua o caminho abaixo pela pasta exata onde o arquivo está no seu Google Drive
# Exemplo: '/content/drive/MyDrive/Seu_Diretorio/custo_medico.csv'
caminho_arquivo = '/content/drive/MyDrive/AulaPUC/2026/Machine Learning LLM/Pratica02/custo_medico.csv'
dados = pd.read_csv(caminho_arquivo)
dados.head()

In [ ]:
dados = pd.read_csv('custo_medico.csv')
dados.head()

In [ ]:
print(f'Formato dos dados: {dados.shape[0]} linhas, {dados.shape[1]} colunas')
dados.info()

In [ ]:
dados.isna().sum()

Boa notícia: esta base não tem nenhum valor faltando, então não precisamos repetir o `dropna()` que fizemos na aula anterior. Vamos direto para conhecer melhor os dados.

In [ ]:
dados.describe()

---

## Etapa 2: Explorando os Dados

Antes de treinar qualquer modelo, vale a pena olhar como a variável que queremos prever (`charges`) se comporta.

In [ ]:
plt.hist(dados['charges'], bins=30, color='#23A8C3', edgecolor='#244357')
plt.xlabel('Custo do plano (US$)')
plt.ylabel('Quantidade de pessoas')
plt.title('Distribuição dos Custos de Plano de Saúde')
plt.grid(alpha=0.3, axis='y')
plt.show()

A distribuição é bem assimétrica: a maioria das pessoas tem custo baixo, mas existe uma cauda de custos bem altos. Vamos ver se `smoker` (fumante) ajuda a explicar essa cauda.

In [ ]:
dados.boxplot(column='charges', by='smoker', figsize=(6, 5))
plt.suptitle('')
plt.title('Custo do Plano: Fumantes vs. Não Fumantes')
plt.xlabel('Fumante?')
plt.ylabel('Custo do plano (US$)')
plt.show()

A diferença é enorme: fumantes custam, em média, quase 4 vezes mais do que não fumantes. É bem provável que `smoker` seja a variável mais importante do nosso modelo, vamos confirmar isso mais adiante com a importância de features.

---

## Etapa 3: Preparando os Dados

Os modelos de Machine Learning só entendem números, mas temos três colunas de texto: `sex`, `smoker` e `region`. Esta etapa também já vem pronta.

Para `sex` e `smoker`, que só têm duas opções cada, basta trocar por 0 e 1. Para `region`, que tem quatro opções, usamos `pd.get_dummies`, que cria uma coluna binária para cada região.

In [ ]:
dados_prontos = dados.copy()

dados_prontos['sex'] = dados_prontos['sex'].map({'female': 0, 'male': 1})
dados_prontos['smoker'] = dados_prontos['smoker'].map({'no': 0, 'yes': 1})

dados_prontos = pd.get_dummies(dados_prontos, columns=['region'], drop_first=True)

dados_prontos.head()

In [ ]:
X = dados_prontos.drop(columns=['charges'])
y = dados_prontos['charges']

print('Features usadas:', list(X.columns))

---

## Etapa 4: Separando Treino e Teste

Mesmo processo de sempre: uma parte dos dados para o modelo aprender, outra para testarmos de forma justa, sem o modelo ter visto essas pessoas antes.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Exemplos de treino: {len(X_treino)}')
print(f'Exemplos de teste: {len(X_teste)}')

Daqui em diante, é com vocês. Cada exercício segue sempre o mesmo padrão que vimos na aula teórica: criar o modelo, treinar com `.fit(...)`, prever com `.predict(...)` e avaliar com as métricas de erro.

---

## Etapa 5, Exercício 1: Regressão Linear

Vamos treinar o modelo mais simples primeiro. Sigam os passos abaixo e completem o código na célula seguinte, nos trechos marcados `# COMPLETE AQUI`.

**Passo a passo:**

1. Crie um dicionário vazio chamado `resultados`, vamos usá-lo para guardar as métricas de cada modelo e comparar todos no final.
2. Crie o modelo de Regressão Linear com `LinearRegression()` e guarde em uma variável chamada `modelo_linear`.
3. Treine o modelo chamando `.fit(X_treino, y_treino)`.
4. Use `.predict(X_teste)` para gerar as previsões no conjunto de teste, e guarde em `y_previsto_linear`.
5. Calcule o MAE, o RMSE e o R² comparando `y_teste` com `y_previsto_linear`. Lembrem-se: RMSE é a raiz quadrada do MSE.
6. Imprima as três métricas.
7. Guarde os resultados em `resultados['Regressão Linear']`, seguindo o mesmo formato que vimos em aula: um dicionário com as chaves `'MAE'`, `'RMSE'` e `'R2'`.

In [ ]:
resultados = {}

# 1. Crie o modelo de Regressão Linear
modelo_linear = None  # COMPLETE AQUI

# 2. Treine o modelo com os dados de treino
# COMPLETE AQUI

# 3. Faça as previsões no conjunto de teste
y_previsto_linear = None  # COMPLETE AQUI

# 4. Calcule as métricas de erro
mae_linear = None  # COMPLETE AQUI
rmse_linear = None  # COMPLETE AQUI
r2_linear = None  # COMPLETE AQUI

print(f'MAE  = US$ {mae_linear:,.2f}')
print(f'RMSE = US$ {rmse_linear:,.2f}')
print(f'R2   = {r2_linear:.2f}')

# 5. Guarde os resultados no dicionário resultados
resultados['Regressão Linear'] = None  # COMPLETE AQUI

💡 **Dica, caso fiquem travados:** o modelo se cria só com `LinearRegression()`, sem nenhum parâmetro. As funções de métrica são `mean_absolute_error(y_real, y_previsto)`, `mean_squared_error(y_real, y_previsto)` (para o RMSE, usem `np.sqrt(...)` em cima do resultado) e `r2_score(y_real, y_previsto)`. Todas elas recebem primeiro o valor real, depois o valor previsto.

---

## Etapa 6, Exercício 2: Árvore de Decisão

Agora repitam o mesmo processo do Exercício 1, mas usando uma Árvore de Decisão em vez de Regressão Linear. O padrão é sempre o mesmo: criar, treinar, prever, avaliar, guardar.

**Passo a passo:**

1. Crie o modelo com `DecisionTreeRegressor(max_depth=5, random_state=42)` e guarde em `modelo_arvore`. Usamos `max_depth=5` para começar, sintam-se à vontade para testar outros valores depois.
2. Treine o modelo.
3. Faça as previsões no conjunto de teste e guarde em `y_previsto_arvore`.
4. Calcule MAE, RMSE e R².
5. Imprima as métricas.
6. Guarde os resultados em `resultados['Árvore de Decisão']`.

In [ ]:
# 1. Crie o modelo de Árvore de Decisão
modelo_arvore = None  # COMPLETE AQUI

# 2. Treine o modelo
# COMPLETE AQUI

# 3. Faça as previsões no conjunto de teste
y_previsto_arvore = None  # COMPLETE AQUI

# 4. Calcule as métricas de erro
mae_arvore = None  # COMPLETE AQUI
rmse_arvore = None  # COMPLETE AQUI
r2_arvore = None  # COMPLETE AQUI

print(f'MAE  = US$ {mae_arvore:,.2f}')
print(f'RMSE = US$ {rmse_arvore:,.2f}')
print(f'R2   = {r2_arvore:.2f}')

# 5. Guarde os resultados no dicionário resultados
resultados['Árvore de Decisão'] = None  # COMPLETE AQUI

💡 **Dica:** repare que este exercício pede exatamente os mesmos passos do Exercício 1, só muda a primeira linha (o modelo criado) e os nomes das variáveis. Se o Exercício 1 funcionou, é só seguir o mesmo padrão aqui.

**Pergunta para pensar:** o R² da Árvore de Decisão foi maior, menor ou igual ao da Regressão Linear? Faz sentido com o que vimos na aula teórica sobre cada um desses modelos?

In [ ]:
importancias = pd.Series(modelo_arvore.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True)

plt.barh(importancias.index, importancias.values, color='#23A8C3', edgecolor='#244357')
plt.xlabel('Importância')
plt.title('Quais Features Mais Influenciam o Custo do Plano?')
plt.grid(alpha=0.3, axis='x')
plt.show()

Confirma a intuição que tivemos no boxplot lá na Etapa 2: `smoker` costuma aparecer como a feature mais importante, de longe.

---

## Etapa 7, Desafio: Random Forest

Este último é por sua conta, sem scaffold de código, só o passo a passo. Se travarem, voltem aos dois exercícios anteriores: o padrão é sempre o mesmo, só muda o modelo.

**Passo a passo:**

1. Crie o modelo com `RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)`.
2. Treine o modelo.
3. Faça as previsões no conjunto de teste.
4. Calcule MAE, RMSE e R².
5. Imprima as métricas.
6. Guarde os resultados em `resultados['Random Forest']`.

Escrevam o código do zero na célula abaixo.

In [ ]:
# Desafio: escreva aqui o código completo para treinar e avaliar o Random Forest


---

## Comparação Final

Esta última parte já vem pronta. Ela lê o dicionário `resultados` que vocês foram preenchendo em cada exercício e monta a tabela e o gráfico de comparação, exatamente como fizemos na aula anterior.

Se algum exercício não foi completado, ele simplesmente não aparece aqui, então voltem e completem todos antes de rodar esta parte.

In [ ]:
tabela_resultados = pd.DataFrame(resultados).T
tabela_resultados = tabela_resultados.sort_values('R2', ascending=False)
tabela_resultados

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(tabela_resultados.index, tabela_resultados['R2'], color='#23A8C3', edgecolor='#244357')
plt.xlabel('R²')
plt.title('Comparação de Modelos: Custo do Plano de Saúde')
plt.xlim(0, 1)
plt.grid(alpha=0.3, axis='x')
plt.show()

**Para refletir:**

1. Qual modelo teve o melhor R²? O resultado te surpreendeu?
2. O MAE está em dólares, o que significa, na prática, um MAE de alguns milhares de dólares para prever o custo de um plano de saúde?
3. Desafio extra: tente `max_depth=3` e `max_depth=15` na Árvore de Decisão e no Random Forest. O que acontece com o R² no teste em cada caso? Isso tem a ver com o que vimos sobre overfitting na aula teórica?
4. Desafio extra: remova a coluna `smoker` de `X` e treine os modelos de novo. O quanto o R² piora? Isso mostra a importância dessa única variável.